# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets and their fields, including their `@id`.


In [ ]:
# Retrieve all record sets' @id and field information
record_sets = dataset.metadata.record_sets
print(f"Record Sets (using @id):\n{'-'*50}")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}  Name: {rs.get('name', '')}")
    print("Fields:")
    for field in rs.get('field', []):
        # Each 'field' is a dict or '@id' reference. Try to show both.
        if isinstance(field, dict):
            print(f"  - Field @id: {field.get('@id','')} (name: {field.get('name','')})")
        else:
            print(f"  - Field @id: {field}")
    print('-'*40)
if len(record_sets) > 0:
    sample_rs_id = record_sets[0]['@id']
    print(f"\nSample records from first record set (@id={sample_rs_id}):")
    for i, rec in enumerate(dataset.records(record_set=sample_rs_id)):
        print(rec)
        if i >= 2: break

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data from record set {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns found: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or categorizing data.

> For demonstration, we pick the first available record set and the first numeric field.

In [ ]:
import numpy as np

# Choose the primary record set to analyze (if available)
if len(dataframes) > 0:
    main_rs_id = list(dataframes.keys())[0]
    df = dataframes[main_rs_id]
    print(f"Primary record set: {main_rs_id}")

    # Identify numeric fields in the DataFrame
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        print(f"Analyzing numeric field: {numeric_field}")

        threshold = df[numeric_field].quantile(0.9)  # Use upper quantile for demonstration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the selected numeric column
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field, if possible
        remaining_fields = [col for col in df.columns if col != numeric_field]
        group_field = None
        if remaining_fields:
            for col in remaining_fields:
                if df[col].dtype in [object, 'category']:
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields available for analysis.")
else:
    print("No dataframes available to analyze.")

## 5. Visualization
Visualize distributions or relationships between key fields in the primary record set.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution/histogram of the main numeric field
if len(dataframes) > 0 and len(numeric_cols) > 0:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {main_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If there is a group_field identified, generate a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
We successfully loaded the Croissant dataset, viewed metadata and structural details using global `@id` references, and demonstrated how to extract, filter, normalize, and visualize numeric and categorical data via `mlcroissant`.

*Key lessons:*
- Use Croissant `@id` fields for all schema entity references to avoid ambiguity.
- Structure code to be dynamic and exploratory, as Croissant datasets can present diverse schemas.
- Data from mass spectrometry or omics workflows often require additional normalization/EDA steps to extract biological or analytical meaning.

Continue your analysis by exploring protein-specific attributes, comparing across categorical variables, or exporting results for further biological knowledge discovery.